# Predicción de Consumo de Alcohol — Rusia 2024

Red neuronal profunda para predecir el consumo de alcohol puro per cápita en regiones de Rusia.

▶ **Ejecutar `Entorno → Tiempo de ejecución → Ejecutar todo`** — no requiere configuración.

## 1. Instalar dependencias

In [ ]:
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn
print("Dependencias listas")

## 2. Importar librerías

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
print("Importaciones listas")

## 3. Definir clases del pipeline

Todas las clases necesarias encapsuladas aquí.

In [ ]:
class CSVLoader:
    @staticmethod
    def load(url):
        encodings = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]
        for enc in encodings:
            try:
                df = pd.read_csv(url, encoding=enc)
                break
            except (UnicodeDecodeError, UnicodeError):
                continue
        else:
            df = pd.read_csv(url, encoding="latin-1", encoding_errors="replace")
        df["Type of alcoholic beverages"] = (
            df["Type of alcoholic beverages"]
            .str.replace(r"[^A-Za-z]ider", "Cider", regex=True)
        )
        return df


class DataPreprocessor:
    def __init__(self):
        self.cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        self.scaler = StandardScaler()
        self.cat_cols_ = ["Region", "Type of alcoholic beverages"]
        self.num_col_count_ = None
        self.fitted_ = False

    def pivot_data(self, df):
        df = df.rename(columns={
            "Consumption of alcoholic beverages (thousands of decaliters)": "vol",
            "Consumption of alcoholic beverages (in liters per capita)": "cap",
            "Consumption of alcoholic beverages (in liters of pure alcohol per capita)": "alc",
        })
        pivoted = df.pivot_table(
            index=["Region", "Type of alcoholic beverages"],
            columns="Year",
            values=["vol", "cap", "alc"],
        )
        pivoted.columns = [f"{metric}_{year}" for metric, year in pivoted.columns]
        pivoted = pivoted.reset_index()
        return pivoted

    def extract_train_data(self, pivoted):
        feature_cols = [f"{m}_{y}" for m in ["vol", "cap", "alc"] for y in range(2017, 2023)]
        X = pivoted[self.cat_cols_ + feature_cols]
        y = pivoted["alc_2023"]
        return X, y

    def extract_predict_data(self, pivoted):
        feature_cols = [f"{m}_{y}" for m in ["vol", "cap", "alc"] for y in range(2018, 2024)]
        X = pivoted[self.cat_cols_ + feature_cols]
        return X

    def _get_num_cols(self, X):
        return [c for c in X.columns if c not in self.cat_cols_]

    def fit(self, X):
        num_cols = self._get_num_cols(X)
        self.cat_encoder.fit(X[self.cat_cols_].values)
        self.scaler.fit(X[num_cols].values)
        self.num_col_count_ = len(num_cols)
        self.fitted_ = True

    def transform(self, X):
        cat_encoded = self.cat_encoder.transform(X[self.cat_cols_].values)
        num_cols = self._get_num_cols(X)
        num_scaled = self.scaler.transform(X[num_cols].values)
        return np.hstack([cat_encoded, num_scaled])

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


class DataSplitter:
    def __init__(self, test_size=0.15, val_size=0.15, random_state=42):
        self.test_size = test_size
        self.val_size = val_size
        self.random_state = random_state

    def split(self, X, y):
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state
        )
        val_frac = self.val_size / (1 - self.test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_frac, random_state=self.random_state
        )
        return {
            "X_train": X_train, "X_val": X_val, "X_test": X_test,
            "y_train": y_train, "y_val": y_val, "y_test": y_test,
            "train_idx": X_train.index,
            "val_idx": X_val.index,
            "test_idx": X_test.index,
        }


class AlcoholDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


class AlcoholPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float("inf")
        self.best_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


class Trainer:
    def __init__(self, model, train_dataset, val_dataset,
                 lr=0.001, weight_decay=1e-5, batch_size=32, max_epochs=200, patience=15):
        self.model = model
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=batch_size)
        self.optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.MSELoss()
        self.early_stopping = EarlyStopping(patience=patience)
        self.max_epochs = max_epochs

    def train(self):
        history = {"train_loss": [], "val_loss": []}
        for epoch in range(1, self.max_epochs + 1):
            train_loss = self._run_epoch(training=True)
            val_loss = self._run_epoch(training=False)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            if epoch % 10 == 0 or epoch == 1:
                print(f"Epoch {epoch:3d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")
            if val_loss == min(history["val_loss"]):
                self.best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            if self.early_stopping(val_loss, self.model):
                print(f"Early stopping at epoch {epoch}")
                break
        self.early_stopping.restore(self.model)
        return history

    def _run_epoch(self, training):
        self.model.train(training)
        loader = self.train_loader if training else self.val_loader
        total_loss = 0.0
        with torch.set_grad_enabled(training):
            for X_batch, y_batch in loader:
                if training:
                    self.optimizer.zero_grad()
                pred = self.model(X_batch)
                loss = self.criterion(pred, y_batch)
                if training:
                    loss.backward()
                    self.optimizer.step()
                total_loss += loss.item()
        return total_loss / len(loader)


class Evaluator:
    def __init__(self):
        self.metrics = {
            "MSE": mean_squared_error,
            "MAE": mean_absolute_error,
            "R²": r2_score,
        }

    def evaluate(self, model, dataset):
        loader = DataLoader(dataset, batch_size=64)
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in loader:
                all_preds.append(model(X_batch).numpy())
                all_true.append(y_batch.numpy())
        y_pred = np.vstack(all_preds).flatten()
        y_true = np.vstack(all_true).flatten()
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}

    def evaluate_sklearn(self, model, X_test, y_test):
        y_pred = model.predict(X_test)
        return {name: metric(y_test, y_pred) for name, metric in self.metrics.items()}


class Predictor:
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model

    def predict_2024(self, pivoted):
        X_2024 = self.preprocessor.extract_predict_data(pivoted)
        X_processed = self.preprocessor.transform(X_2024)
        dataset = AlcoholDataset(X_processed)
        loader = DataLoader(dataset, batch_size=64)
        self.model.eval()
        predictions = []
        with torch.no_grad():
            for X_batch in loader:
                predictions.append(self.model(X_batch).numpy())
        y_pred = np.vstack(predictions).flatten()
        y_pred = np.maximum(y_pred, 0)
        result = pivoted[["Region", "Type of alcoholic beverages"]].copy()
        result["Predicted_2024"] = y_pred
        return result

    def ranking_by_region(self, predictions):
        return (
            predictions.groupby("Region")["Predicted_2024"]
            .mean().sort_values(ascending=False)
            .reset_index()
            .rename(columns={"Predicted_2024": "Avg_Pure_Alcohol_2024"})
        )

    def ranking_by_beverage(self, predictions):
        return (
            predictions.groupby("Type of alcoholic beverages")["Predicted_2024"]
            .mean().sort_values(ascending=False)
            .reset_index()
            .rename(columns={"Predicted_2024": "Avg_Pure_Alcohol_2024"})
        )


print("Clases definidas")

## 4. Cargar datos desde GitHub

In [ ]:
URL = "https://raw.githubusercontent.com/jfelipeq14/alcohol-prediction-russia/main/Consumption%20of%20alcoholic%20beverages%202017-2023%20(Pivot%20table).csv"
df = CSVLoader.load(URL)
print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
print(f"Tipos de bebida: {list(df['Type of alcoholic beverages'].unique())}")
print(f"Regiones: {df['Region'].nunique()}")
print(f"Años: {sorted(df['Year'].unique())}")

## 5. Preprocesar (pivot, split, encode, scale)

In [ ]:
preprocessor = DataPreprocessor()
pivoted = preprocessor.pivot_data(df)
print(f"Muestras (región × bebida): {len(pivoted)}")

X, y = preprocessor.extract_train_data(pivoted)
print(f"Features: {X.shape[1]} | Target: {y.name}")

splitter = DataSplitter()
splits = splitter.split(X, y)
print(f"Train: {len(splits['X_train'])} | Val: {len(splits['X_val'])} | Test: {len(splits['X_test'])}}")

X_train = preprocessor.fit_transform(splits["X_train"])
X_val = preprocessor.transform(splits["X_val"])
X_test = preprocessor.transform(splits["X_test"])
y_train = splits["y_train"].values
y_val = splits["y_val"].values
y_test = splits["y_test"].values

train_dataset = AlcoholDataset(X_train, y_train)
val_dataset = AlcoholDataset(X_val, y_val)
test_dataset = AlcoholDataset(X_test, y_test)

input_dim = X_train.shape[1]
print(f"Dimensión de entrada: {input_dim}")

## 6. Construir y entrenar red neuronal

In [ ]:
model = AlcoholPredictor(input_dim=input_dim, hidden_dims=[128, 64], dropout=0.3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Arquitectura: {input_dim} → [128, 64] → 1")
print(f"Parámetros: {total_params:,}")

trainer = Trainer(model, train_dataset, val_dataset,
                  lr=0.001, weight_decay=1e-5, batch_size=32,
                  max_epochs=200, patience=15)
history = trainer.train()
print(f"\nMejor validation MSE: {min(history['val_loss']):.6f}")

## 7. Evaluar (NN vs Linear Regression)

In [ ]:
evaluator = Evaluator()
nn_metrics = evaluator.evaluate(model, test_dataset)
print("── Neural Network ──")
for name, val in nn_metrics.items():
    print(f"  {name}: {val:.4f}")

baseline = LinearRegression()
baseline.fit(X_train, y_train)
base_metrics = evaluator.evaluate_sklearn(baseline, X_test, y_test)
print("\n── Linear Regression ──")
for name, val in base_metrics.items():
    print(f"  {name}: {val:.4f}")

## 8. Predecir 2024

In [ ]:
predictor = Predictor(preprocessor, model)
predictions = predictor.predict_2024(pivoted)

region_ranking = predictor.ranking_by_region(predictions)
beverage_ranking = predictor.ranking_by_beverage(predictions)

print(f"Predicciones: {len(predictions)} filas")
display(predictions.head(10))

print("\n── Ranking por región (top 10) ──")
display(region_ranking.head(10))

print("\n── Ranking por bebida ──")
display(beverage_ranking)

## 9. Comparación 2023 vs 2024

Merge de `alc_2023` (real) con `Predicted_2024` para evaluar cambios estimados.

In [ ]:
comp = pivoted[["Region", "Type of alcoholic beverages", "alc_2023"]].copy()
comp = comp.merge(predictions, on=["Region", "Type of alcoholic beverages"])
comp["Diferencia"] = comp["Predicted_2024"] - comp["alc_2023"]
comp["Dif_%"] = (comp["Diferencia"] / comp["alc_2023"].replace(0, np.nan)) * 100

region_change = comp.groupby("Region").agg(
    Real_2023=("alc_2023", "mean"),
    Pred_2024=("Predicted_2024", "mean"),
    Diferencia=("Diferencia", "mean"),
).sort_values("Diferencia", ascending=False)

up = (region_change["Diferencia"] > 0).sum()
down = (region_change["Diferencia"] <= 0).sum()
print(f"Regiones con aumento estimado: {up} | Disminución estimada: {down}\n")

print("Top 10 regiones — mayor aumento estimado:")
display(region_change.head(10).style.format("{:.4f}"))

print("Top 10 regiones — mayor disminución estimada:")
display(region_change.tail(10).sort_values("Diferencia").style.format("{:.4f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(comp["alc_2023"], comp["Predicted_2024"],
           alpha=0.4, edgecolors="k", linewidth=0.5)
lims = [0, comp[["alc_2023", "Predicted_2024"]].max().max() + 0.5]
ax.plot(lims, lims, "r--", alpha=0.5, label="Sin cambio")
ax.set_xlabel("Real 2023")
ax.set_ylabel("Predicho 2024")
ax.set_title("Comparación 2023 vs 2024")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Gráficas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history["train_loss"], label="Train MSE", alpha=0.8)
ax.plot(history["val_loss"], label="Val MSE", alpha=0.8)
best_epoch = np.argmin(history["val_loss"]) + 1
ax.axvline(best_epoch - 1, color="gray", linestyle="--", alpha=0.5, label=f"Mejor época ({best_epoch})")
ax.set_xlabel("Época")
ax.set_ylabel("MSE")
ax.set_title("Historial de entrenamiento")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_test_pred_nn = model(torch.tensor(X_test, dtype=torch.float32)).detach().numpy().flatten()
y_test_pred_base = baseline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_pred, title in zip(
    axes,
    [y_test_pred_nn, y_test_pred_base],
    ["Neural Network", "Linear Regression"],
):
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidth=0.5)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", alpha=0.7)
    ax.set_xlabel("Real")
    ax.set_ylabel("Predicho")
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_n = 15
region_ranking.head(top_n).set_index("Region").plot(
    kind="bar", ax=axes[0], color="steelblue", legend=False
)
axes[0].set_title(f"Top {top_n} regiones — Consumo estimado 2024")
axes[0].set_ylabel("Litros per cápita")
axes[0].tick_params(axis="x", rotation=45)

region_ranking.tail(top_n).set_index("Region").plot(
    kind="bar", ax=axes[1], color="coral", legend=False
)
axes[1].set_title(f"Bottom {top_n} regiones — Consumo estimado 2024")
axes[1].set_ylabel("Litros per cápita")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.Set2(np.linspace(0, 1, len(beverage_ranking)))
ax.barh(beverage_ranking["Type of alcoholic beverages"],
        beverage_ranking["Avg_Pure_Alcohol_2024"], color=colors)
ax.set_xlabel("Litros de alcohol puro per cápita")
ax.set_title("Consumo estimado 2024 por tipo de bebida")
for i, v in enumerate(beverage_ranking["Avg_Pure_Alcohol_2024"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(nn_metrics))
nn_vals = list(nn_metrics.values())
base_vals = list(base_metrics.values())
ax.bar(x, nn_vals, width=0.35, label="Neural Network", color="steelblue", alpha=0.8)
ax.bar([i + 0.35 for i in x], base_vals, width=0.35, label="Linear Regression", color="coral", alpha=0.8)
ax.set_xticks([i + 0.175 for i in x])
ax.set_xticklabels(list(nn_metrics.keys()))
ax.set_title("Comparación de métricas")
ax.legend()
for bars in [nn_vals, base_vals]:
    for i, v in enumerate(bars):
        offset = 0.175 if bars is nn_vals else 0.525
        ax.text(i + offset - 0.175, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 11. Análisis de residuales

In [ ]:
residuals = y_test - y_test_pred_nn

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_test, bins=20, alpha=0.6, label="Real", color="steelblue", edgecolor="white")
axes[0].hist(y_test_pred_nn, bins=20, alpha=0.6, label="Predicho NN", color="coral", edgecolor="white")
axes[0].set_xlabel("Litros de alcohol puro per cápita")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución: Real vs Predicho")
axes[0].legend()

axes[1].scatter(y_test_pred_nn, residuals, alpha=0.6, edgecolors="k", linewidth=0.5)
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Predicho")
axes[1].set_ylabel("Residual")
axes[1].set_title("Gráfico de residuales")

plt.tight_layout()
plt.show()

In [ ]:
pivoted_test = pivoted.loc[splits["test_idx"]]
beverage_types_test = pivoted_test["Type of alcoholic beverages"].values

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=beverage_types_test, y=residuals, ax=ax, palette="Set2")
ax.axhline(0, color="red", linestyle="--", alpha=0.4)
ax.set_title("Residuales por tipo de bebida")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 12. Correlaciones entre años

In [ ]:
year_cols = [f"alc_{y}" for y in range(2017, 2024)]
corr_data = pivoted[year_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title("Correlación entre años")
plt.tight_layout()
plt.show()

In [ ]:
print("=== Fin ===")
print("\n💡 Los resultados solo están disponibles durante esta sesión de Colab.")
print("   Para descargarlos: panel Archivos (📁) → download.")